# BBO Capstone: Method and Results

This notebook summarises the **black-box optimisation** workflow used in the capstone: accumulate evaluations \((\mathbf{x}, y)\) from an external simulator, fit a **Gaussian Process** surrogate per function, and choose the next query using **Expected Improvement** (see `src/capstone_manager.py`).

**How to run:** from the repository root, activate your environment (`pip install -r requirements.txt`), then open this notebook. Paths below assume the working directory is `notebooks/` or adjust `ROOT`.

## 1. Method (high level)

- **Inputs:** $\mathbf{x} \in [0,1]^d$ (dimension $d$ depends on the function).
- **Output:** scalar $y$ from the unknown objective (maximise or minimise per `data/metadata.json`).
- **Surrogate:** GP with Matérn kernel (`sklearn.gaussian_process`).
- **Acquisition:** Expected Improvement with exploration parameter $\xi$; candidates sampled uniformly, best EI selected.
- **Reproducibility:** Round results can be imported as CSV and history rebuilt via `sync-history` (see README).

In [ ]:
from pathlib import Path
import csv
import json

# Repository root (parent of notebooks/)
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

meta_path = ROOT / "data" / "metadata.json"
meta = json.loads(meta_path.read_text(encoding="utf-8"))
print("Function specs:")
for f in meta["functions"]:
    print(f"  F{f['function_id']}: dims={f['dims']}, objective={f['objective']}")

In [ ]:
def function_row_count(function_id: int) -> int:
    path = ROOT / "data" / "functions" / f"function_{function_id}.csv"
    with path.open(newline="", encoding="utf-8") as fp:
        return max(0, sum(1 for _ in fp) - 1)

rows = {fid: function_row_count(fid) for fid in range(1, 9)}
print("Observations per function (rows in data/functions/function_*.csv):")
for fid, n in rows.items():
    print(f"  F{fid}: {n}")

## 2. Results (local store)

The leaderboard and final scores are produced on the **course evaluation platform** (not stored in this repo). This repository keeps **local copies** of submitted queries and imported results for reproducibility. Update `data/history/` with exported round CSVs and run `python src/capstone_manager.py sync-history` to align `data/functions/` with your official history.

See **docs/MODEL_CARD.md** for performance notes and limitations.